# Module 09: GraphRAG & Knowledge Graphs,

focusing on the ingestion and extraction pipeline using Python libraries like NetworkX and enterprise graph storage with Neo4j.

## Part 1: The Knowledge Graph Extraction Pipeline (Text $\rightarrow$ Triples)

Unlike standard RAG, which breaks documents into independent chunks, GraphRAG converts unstructured text into structural semantic components known as Triples: (Subject, Predicate, Object).

### 1. The Extraction Concept
Given a sentence:

"Alice works as the Lead Engineer at TechCorp, which partners with DataStream in Berlin."

An LLM extraction prompt transforms this into structured graph components:

Nodes (Entities): Alice (Person), TechCorp (Organization), DataStream (Organization), Berlin (Location).

Edges (Relationships):

(Alice) -[:WORKS_AT {role: "Lead Engineer"}]-> (TechCorp)

(TechCorp) -[:PARTNERS_WITH]-> (DataStream)

(TechCorp) -[:LOCATED_IN]-> (Berlin)

## Part 2: Prototyping Locally with NetworkX
For local experimentation and building light schemas in your repository, NetworkX allows you to manipulate graphs entirely in-memory.

Complete Extraction & Traversal Script (01_networkx_graph_builder.py)

In [ ]:
import networkx as nx

class LocalGraphRAG:
    def __init__(self):
        # Initialize a directed graph
        self.graph = nx.DiGraph()

    def add_triples(self, triples):
        """
        triples: list of tuples like (subject, relation, object, metadata_dict)
        """
        for subj, rel, obj, meta in triples:
            # Add nodes with basic property attributes
            self.graph.add_node(subj, entity_type=meta.get("subj_type", "Unknown"))
            self.graph.add_node(obj, entity_type=meta.get("obj_type", "Unknown"))
            
            # Add directed edge with relationship type
            self.graph.add_edge(subj, obj, relation=rel)

    def multi_hop_traverse(self, start_entity: str, max_hops: int = 2):
        """Traverses the graph up to N-hops to gather connected contexts."""
        subgraph_nodes = set([start_entity])
        current_layer = set([start_entity])
        
        for hop in range(max_hops):
            next_layer = set()
            for node in current_layer:
                # Get outgoing neighbors
                for neighbor in self.graph.successors(node):
                    edge_data = self.graph.get_edge_data(node, neighbor)
                    print(f"Hop {hop+1}: ({node}) -[{edge_data['relation']}]-> ({neighbor})")
                    subgraph_nodes.add(neighbor)
                    next_layer.add(neighbor)
            current_layer = next_layer
            
        return subgraph_nodes

# --- Example Usage ---
if __name__ == "__main__":
    kg = LocalGraphRAG()
    
    # Simulate extracted triples from text parsing
    sample_triples = [
        ("Alice", "WORKS_AT", "TechCorp", {"subj_type": "Person", "obj_type": "Organization"}),
        ("TechCorp", "PARTNERS_WITH", "DataStream", {"subj_type": "Organization", "obj_type": "Organization"}),
        ("DataStream", "LOCATED_IN", "Berlin", {"subj_type": "Organization", "obj_type": "Place"})
    ]
    
    kg.add_triples(sample_triples)
    
    print("Executing Multi-Hop Graph Traversal from 'Alice':")
    connected = kg.multi_hop_traverse("Alice", max_hops=2)
    print(f"Retrieved Subgraph Entities: {list(connected)}")

## Part 3: Scaling to Production with Neo4j & Official Drivers
When your corpus grows beyond memory limitations, NetworkX is replaced by graph databases like Neo4j using the official neo4j-graphrag package.

### 1. Connecting and Initializing the Neo4j Pipeline

In [ ]:
from neo4j import GraphDatabase
from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings.openai import OpenAIEmbeddings
from neo4j_graphrag.retrievers import VectorRetriever

# 1. Establish Driver Connection
uri = "neo4j://localhost:7687"
auth = ("neo4j", "password_secure")
driver = GraphDatabase.driver(uri, auth=auth)

# 2. Configure Foundation LLM and Embedder for Extraction & Search
llm = OpenAILLM(model_name="gpt-4o-mini", model_params={"temperature": 0})
embedder = OpenAIEmbeddings(model="text-embedding-3-small")

### 2. Querying with Cypher Retrieval Patterns
Instead of calculating text distances blindly, Neo4j uses structural query statements (Cypher) to retrieve precise contextual neighborhoods:

In [ ]:
// Cypher query to pull multi-hop context for an LLM prompt
MATCH (p:Person {name: "Alice"})-[:WORKS_AT]->(org:Organization)
OPTIONAL MATCH (org)-[:PARTNERS_WITH]->(partner:Organization)
RETURN p.name AS Person, org.name AS Company, collect(partner.name) AS Partners

## Part 4: Production Challenges & Solutions

Entity Resolution / Normalization: Different text chunks might refer to the same entity under alternate names (e.g., "Microsoft", "MSFT", "Microsoft Corporation"). Run a deduplication or canonical mapping step during triple storage.

Token Window Overload: Graph paths can expand exponentially. Use subgraph summarization or limit traversal depth to $k \le 2$ hops to avoid context window inflation.GraphRAG Tutorial with Neo4j & PythonThis video provides a helpful visual walkthrough of extracting entities from financial data and mapping them into a connected Neo4j knowledge graph.

GraphRAG Tutorial with Neo4j & Python[https://www.youtube.com/watch?v=bXskejJDBoA]

This video provides a helpful visual walkthrough of extracting entities from financial data and mapping them into a connected Neo4j knowledge graph